In [3]:
# Complete Manipuri Emotion Detection System
# Optimized for your dataset at /content/manipuri_emotion_dataset_main1.xlsx

import pandas as pd
import numpy as np
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack, csr_matrix

# Advanced ML Libraries
import xgboost as xgb
import lightgbm as lgb
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# For text processing
import sys
try:
    from indicnlp.tokenize import indic_tokenize
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    INDIC_AVAILABLE = True
except:
    INDIC_AVAILABLE = False
    print("Note: Indic NLP library not available. Using basic tokenization.")
    print("Install with: !pip install indic-nlp-library")

# ===================== 1. LOAD AND EXPLORE YOUR DATASET =====================

def load_dataset(file_path):
    """Load and explore your Manipuri emotion dataset"""
    print("="*60)
    print("LOADING MANIPURI EMOTION DATASET")
    print("="*60)
    
    # Load the Excel file
    df = pd.read_excel(file_path)
    
    print(f"\nDataset loaded successfully!")
    print(f"Total samples: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Display first few rows
    print("\nFirst 5 samples:")
    print(df.head())
    
    # Check for missing values
    print("\nMissing values:")
    print(df.isnull().sum())
    
    # Assuming column names - modify if different
    # Try to identify text and emotion columns
    text_col = None
    emotion_col = None
    
    for col in df.columns:
        if 'text' in col.lower() or 'sentence' in col.lower() or 'manipuri' in col.lower():
            text_col = col
        if 'emotion' in col.lower() or 'label' in col.lower() or 'sentiment' in col.lower():
            emotion_col = col
    
    if text_col is None:
        text_col = df.columns[0]  # Assume first column is text
    if emotion_col is None:
        emotion_col = df.columns[1]  # Assume second column is emotion
    
    print(f"\nUsing:")
    print(f"Text column: {text_col}")
    print(f"Emotion column: {emotion_col}")
    
    # Rename for consistency
    df = df.rename(columns={text_col: 'text', emotion_col: 'emotion'})
    
    # Remove any rows with missing values
    df = df.dropna(subset=['text', 'emotion'])
    
    # Clean text (remove extra spaces, etc.)
    df['text'] = df['text'].astype(str).str.strip()
    df['emotion'] = df['emotion'].astype(str).str.strip()
    
    # Display emotion distribution
    print("\nEmotion Distribution:")
    emotion_counts = df['emotion'].value_counts()
    for emotion, count in emotion_counts.items():
        percentage = (count / len(df)) * 100
        print(f"  {emotion}: {count} ({percentage:.1f}%)")
    
    # Visualize distribution
    plt.figure(figsize=(10, 6))
    emotion_counts.plot(kind='bar')
    plt.title('Emotion Distribution in Dataset')
    plt.xlabel('Emotion')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('emotion_distribution.png')
    plt.show()
    
    return df

# ===================== 2. ADVANCED PREPROCESSING FOR MANIPURI =====================

class ManipuriPreprocessor:
    """Specialized preprocessor for Manipuri text"""
    
    def __init__(self):
        self.normalizer = None
        if INDIC_AVAILABLE:
            try:
                self.normalizer = IndicNormalizerFactory().get_normalizer("mni")
            except:
                pass
        
        # Manipuri stopwords (common words that don't carry emotion)
        self.stopwords = set([
            'ꯑꯩ', 'ꯅꯪ', 'ꯃꯍꯥꯛ', 'ꯑꯩꯉꯣꯟ', 'ꯅꯪꯉꯣꯟ', 'ꯑꯗꯨ', 'ꯃꯁꯤ',
            'ꯃꯐꯝ', 'ꯃꯇꯝ', 'ꯑꯣꯏ', 'ꯅꯠꯇ꯭ꯔꯒ', 'ꯑꯃ', 'ꯑꯅꯤ', 'ꯍꯧꯖꯤꯛ',
            'ꯃꯨꯛ', 'ꯆꯤꯟꯖꯥ', 'ꯃꯥ', 'ꯍꯥ', 'ꯏ', 'ꯐꯥ', 'ꯍꯩ', 'ꯅꯠꯇꯦ',
            'ꯇꯥ', 'ꯄꯨ', 'ꯊꯥ', 'ꯂꯩ', 'ꯇꯝ', 'ꯁꯤ', 'ꯂꯤ', 'ꯀꯤ', 'ꯒꯤ', 'ꯗ'
        ])
    
    def clean_text(self, text):
        """Comprehensive text cleaning"""
        if not isinstance(text, str):
            return ""
        
        # Convert to string and strip
        text = str(text).strip()
        
        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text)
        
        # Remove URLs and mentions
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+', '', text)
        
        # Remove emojis and special characters (keep Manipuri script)
        # Meitei Mayek: U+ABC0-U+ABFF
        # Bengali script: U+0980-U+09FF (often used for Manipuri)
        text = re.sub(r'[^\u0900-\u097F\u0980-\u09FF\u0A00-\u0A7F\uABC0-\uABFF\uAA60-\uAA7F\s\.\,\!\?\-]', '', text)
        
        # Normalize using Indic NLP if available
        if self.normalizer:
            try:
                text = self.normalizer.normalize(text)
            except:
                pass
        
        # Remove extra spaces again
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text.lower()
    
    def get_text_features(self, text):
        """Extract additional text features"""
        words = text.split()
        return {
            'word_count': len(words),
            'char_count': len(text),
            'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
            'unique_words': len(set(words))
        }

# ===================== 3. DATA AUGMENTATION FOR SMALL DATASET =====================

class DataAugmenter:
    """Data augmentation for low-resource Manipuri"""
    
    def __init__(self):
        # Emotion-specific keywords for augmentation
        self.emotion_keywords = {
            'joy': ['ꯍꯛꯄ', 'ꯅꯨꯡꯉꯥꯏ', 'ꯐꯥꯔꯦ', 'ꯄꯥꯝꯂꯤ', 'ꯍꯦꯔꯥꯏ', 'ꯅꯨꯡꯁꯤ', 'ꯌꯥꯝꯅ', 'ꯐꯠꯇꯕ'],
            'sadness': ['ꯊꯥꯈ꯭ꯔꯦ', 'ꯂꯝꯗꯝ', 'ꯅꯨꯡꯁꯤꯗꯕ', 'ꯑꯀꯤ', 'ꯑꯉꯥꯡ', 'ꯐꯠꯇ', 'ꯋꯥꯔꯦ', 'ꯆꯤꯡꯈꯠ'],
            'anger': ['ꯁꯥꯎ', 'ꯀꯣꯞ', 'ꯆꯤꯡꯊꯤ', 'ꯁꯥꯟ', 'ꯇꯥꯔꯦ', 'ꯊꯤꯡꯈꯠ', 'ꯂꯧꯁꯤ', 'ꯆꯦꯠꯄ'],
            'fear': ['ꯂꯤꯔꯤ', 'ꯀꯤꯅꯤ', 'ꯍꯧꯍꯩ', 'ꯆꯦꯠ', 'ꯀꯨꯞ', 'ꯌꯥꯝ', 'ꯀꯥꯡꯖꯩ', 'ꯋꯥꯔꯤ'],
            'love': ['ꯅꯨꯡꯁꯤ', 'ꯄꯥꯝꯂꯤ', 'ꯅꯨꯡꯁꯤꯟ', 'ꯊꯨꯒꯥ', 'ꯅꯨꯡꯉꯥ', 'ꯅꯨꯡꯁꯤꯠ', 'ꯄꯥꯝꯖꯤ'],
            'surprise': ['ꯊꯥꯖ꯭ꯔꯤ', 'ꯁꯣꯛꯄ', 'ꯑꯉꯥꯡ', 'ꯑꯀꯥ', 'ꯆꯦꯠꯄ', 'ꯌꯥꯝꯅ', 'ꯑꯉꯥꯡꯕ'],
            'neutral': ['ꯂꯩ', 'ꯊꯤꯡꯖꯤ', 'ꯇꯝꯕ', 'ꯊꯕꯛ', 'ꯄꯨꯛ', 'ꯃꯐꯝ', 'ꯃꯇꯝ']
        }
    
    def synonym_replacement(self, text, emotion, n_replacements=1):
        """Replace words with emotion synonyms"""
        words = text.split()
        if not words:
            return [text]
        
        keywords = self.emotion_keywords.get(emotion, [])
        if not keywords:
            return [text]
        
        augmented = []
        for _ in range(n_replacements):
            new_words = words.copy()
            idx = np.random.randint(0, len(words))
            new_words[idx] = np.random.choice(keywords)
            augmented.append(' '.join(new_words))
        
        return augmented
    
    def augment_data(self, texts, emotions, factor=2):
        """Augment dataset"""
        augmented_texts = list(texts)
        augmented_emotions = list(emotions)
        
        for text, emotion in zip(texts, emotions):
            # Add synonym variations
            syn_texts = self.synonym_replacement(text, emotion, factor-1)
            augmented_texts.extend(syn_texts)
            augmented_emotions.extend([emotion] * len(syn_texts))
        
        return np.array(augmented_texts), np.array(augmented_emotions)

# ===================== 4. FEATURE ENGINEERING =====================

class FeatureEngineer:
    """Extract rich features from Manipuri text"""
    
    def __init__(self, max_features=10000):
        self.max_features = max_features
        
        # Multiple n-gram features
        self.tfidf_1gram = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 1),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        
        self.tfidf_2gram = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(2, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        
        self.tfidf_3gram = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(3, 3),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
        
        self.char_tfidf = TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 5),
            max_features=5000,
            sublinear_tf=True
        )
        
        self.count_vec = CountVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2
        )
        
        self.feature_selector = SelectKBest(mutual_info_classif, k=8000)
        
    def extract_all_features(self, texts, fit=False):
        """Extract and combine all feature sets"""
        if fit:
            f1 = self.tfidf_1gram.fit_transform(texts)
            f2 = self.tfidf_2gram.fit_transform(texts)
            f3 = self.tfidf_3gram.fit_transform(texts)
            f4 = self.char_tfidf.fit_transform(texts)
            f5 = self.count_vec.fit_transform(texts)
        else:
            f1 = self.tfidf_1gram.transform(texts)
            f2 = self.tfidf_2gram.transform(texts)
            f3 = self.tfidf_3gram.transform(texts)
            f4 = self.char_tfidf.transform(texts)
            f5 = self.count_vec.transform(texts)
        
        # Combine all features
        combined = hstack([f1, f2, f3, f4, f5])
        return combined

# ===================== 5. ENSEMBLE CLASSIFIER =====================

class ManipuriEmotionEnsemble:
    """High-performance ensemble for Manipuri emotion detection"""
    
    def __init__(self):
        self.preprocessor = ManipuriPreprocessor()
        self.feature_engineer = FeatureEngineer()
        self.label_encoder = LabelEncoder()
        
        # XGBoost (best for small datasets)
        self.xgb = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='mlogloss'
        )
        
        # LightGBM
        self.lgb = lgb.LGBMClassifier(
            n_estimators=300,
            max_depth=7,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            class_weight='balanced'
        )
        
        # Random Forest
        self.rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        
        # Gradient Boosting
        self.gb = GradientBoostingClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42
        )
        
        # Logistic Regression
        self.lr = LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        )
        
        # Voting Classifier
        self.voting = VotingClassifier(
            estimators=[
                ('xgb', self.xgb),
                ('lgb', self.lgb),
                ('rf', self.rf),
                ('gb', self.gb)
            ],
            voting='soft',
            weights=[3, 3, 2, 2]
        )
        
        # Stacking Classifier
        self.stacking = StackingClassifier(
            estimators=[
                ('xgb', self.xgb),
                ('lgb', self.lgb),
                ('rf', self.rf)
            ],
            final_estimator=LogisticRegression(C=1.0, max_iter=1000),
            cv=5,
            n_jobs=-1
        )
        
        self.best_model = None
        self.best_model_name = None
        
    def preprocess_texts(self, texts):
        """Clean and preprocess texts"""
        return [self.preprocessor.clean_text(text) for text in texts]
    
    def prepare_features(self, texts, y=None, fit=False):
        """Prepare features for model"""
        cleaned_texts = self.preprocess_texts(texts)
        
        if fit:
            features = self.feature_engineer.extract_all_features(cleaned_texts, fit=True)
            if y is not None:
                features = self.feature_engineer.feature_selector.fit_transform(features, y)
            return features
        else:
            features = self.feature_engineer.extract_all_features(cleaned_texts, fit=False)
            if hasattr(self.feature_engineer.feature_selector, 'scores_'):
                features = self.feature_engineer.feature_selector.transform(features)
            return features
    
    def train_with_cv(self, X, y, cv_folds=5):
        """Train with cross-validation and select best model"""
        print("\n" + "="*60)
        print("CROSS-VALIDATION TRAINING")
        print("="*60)
        
        # Prepare features
        print("Extracting features...")
        X_features = self.prepare_features(X, y, fit=True)
        y_encoded = self.label_encoder.fit_transform(y)
        
        print(f"Features shape: {X_features.shape}")
        print(f"Number of classes: {len(self.label_encoder.classes_)}")
        print(f"Classes: {self.label_encoder.classes_}")
        
        # Define models to evaluate
        models = {
            'XGBoost': self.xgb,
            'LightGBM': self.lgb,
            'Random Forest': self.rf,
            'Gradient Boosting': self.gb,
            'Voting Ensemble': self.voting,
            'Stacking Ensemble': self.stacking
        }
        
        # Cross-validation
        skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        results = {}
        
        for name, model in models.items():
            print(f"\nEvaluating {name}...")
            scores = []
            
            for fold, (train_idx, val_idx) in enumerate(skf.split(X_features, y_encoded), 1):
                X_train_fold = X_features[train_idx]
                X_val_fold = X_features[val_idx]
                y_train_fold = y_encoded[train_idx]
                y_val_fold = y_encoded[val_idx]
                
                # Clone model
                if name in ['XGBoost', 'LightGBM', 'Random Forest', 'Gradient Boosting']:
                    model_clone = model.__class__(**model.get_params())
                else:
                    model_clone = model
                
                model_clone.fit(X_train_fold, y_train_fold)
                score = model_clone.score(X_val_fold, y_val_fold)
                scores.append(score)
                print(f"  Fold {fold}: {score:.4f}")
            
            mean_score = np.mean(scores)
            std_score = np.std(scores)
            results[name] = {'mean': mean_score, 'std': std_score, 'model': model}
            print(f"  Mean CV Score: {mean_score:.4f} (+/- {std_score:.4f})")
        
        # Select best model
        best_name = max(results, key=lambda x: results[x]['mean'])
        self.best_model = results[best_name]['model']
        self.best_model_name = best_name
        
        print(f"\n{'='*60}")
        print(f"Best Model: {best_name}")
        print(f"CV Score: {results[best_name]['mean']:.4f} (+/- {results[best_name]['std']:.4f})")
        print(f"{'='*60}")
        
        # Train best model on full data
        print(f"\nTraining {best_name} on full dataset...")
        self.best_model.fit(X_features, y_encoded)
        
        return self.best_model
    
    def predict(self, texts):
        """Predict emotions"""
        if self.best_model is None:
            raise ValueError("Model not trained yet!")
        
        X_features = self.prepare_features(texts, fit=False)
        predictions = self.best_model.predict(X_features)
        return self.label_encoder.inverse_transform(predictions)
    
    def predict_proba(self, texts):
        """Get prediction probabilities"""
        if self.best_model is None:
            raise ValueError("Model not trained yet!")
        
        X_features = self.prepare_features(texts, fit=False)
        proba = self.best_model.predict_proba(X_features)
        return proba
    
    def evaluate(self, X_test, y_test):
        """Evaluate model performance"""
        y_pred = self.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        f1_macro = f1_score(y_test, y_pred, average='macro')
        
        print("\n" + "="*60)
        print("MODEL EVALUATION RESULTS")
        print("="*60)
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score (Weighted): {f1:.4f}")
        print(f"F1-Score (Macro):    {f1_macro:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.label_encoder.classes_,
                    yticklabels=self.label_encoder.classes_)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.xticks(rotation=45)
        plt.yticks(rotation=45)
        plt.tight_layout()
        plt.savefig('confusion_matrix.png')
        plt.show()
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_weighted': f1,
            'f1_macro': f1_macro,
            'predictions': y_pred,
            'confusion_matrix': cm
        }

# ===================== 6. MAIN TRAINING PIPELINE =====================

def main():
    """Main training pipeline for Manipuri emotion detection"""
    
    # 1. Load your dataset
    dataset_path = '/content/manipuri_emotion_dataset_main1.xlsx'
    df = load_dataset(dataset_path)
    
    # 2. Prepare data
    X = df['text'].values
    y = df['emotion'].values
    
    # 3. Split data (stratified to maintain class distribution)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    print(f"\nData Split:")
    print(f"Training: {len(X_train)} samples")
    print(f"Validation: {len(X_val)} samples")
    print(f"Testing: {len(X_test)} samples")
    
    # 4. Data augmentation (optional - for small datasets)
    if len(X_train) < 2000:
        print("\nApplying data augmentation...")
        augmenter = DataAugmenter()
        X_train_aug, y_train_aug = augmenter.augment_data(X_train, y_train, factor=2)
        print(f"After augmentation: {len(X_train_aug)} training samples")
        X_train, y_train = X_train_aug, y_train_aug
    
    # 5. Train model
    model = ManipuriEmotionEnsemble()
    model.train_with_cv(X_train, y_train, cv_folds=5)
    
    # 6. Evaluate on validation set
    print("\n" + "="*60)
    print("VALIDATION SET EVALUATION")
    print("="*60)
    val_results = model.evaluate(X_val, y_val)
    
    # 7. Final evaluation on test set
    print("\n" + "="*60)
    print("TEST SET FINAL EVALUATION")
    print("="*60)
    test_results = model.evaluate(X_test, y_test)
    
    # 8. Save model
    print("\nSaving model...")
    joblib.dump(model, 'manipuri_emotion_model.pkl')
    
    # Save label encoder separately
    with open('label_encoder.pkl', 'wb') as f:
        pickle.dump(model.label_encoder, f)
    
    print("Model saved as 'manipuri_emotion_model.pkl'")
    
    # 9. Test with sample predictions
    print("\n" + "="*60)
    print("SAMPLE PREDICTIONS")
    print("="*60)
    
    # Get some test samples
    sample_indices = np.random.choice(len(X_test), min(10, len(X_test)), replace=False)
    for idx in sample_indices:
        text = X_test[idx]
        true_label = y_test[idx]
        pred_label = model.predict([text])[0]
        proba = model.predict_proba([text])[0]
        confidence = np.max(proba)
        
        print(f"\nText: {text}")
        print(f"True: {true_label} | Predicted: {pred_label} | Confidence: {confidence:.4f}")
    
    return model, test_results

# ===================== 7. PREDICTION FUNCTION =====================

def predict_emotion(text, model_path='manipuri_emotion_model.pkl'):
    """Function to predict emotion for a single Manipuri text"""
    # Load model
    model = joblib.load(model_path)
    
    # Predict
    emotion = model.predict([text])[0]
    probabilities = model.predict_proba([text])[0]
    
    # Get top 3 emotions
    top_3_idx = np.argsort(probabilities)[-3:][::-1]
    top_3 = [(model.label_encoder.classes_[idx], probabilities[idx]) for idx in top_3_idx]
    
    return {
        'text': text,
        'emotion': emotion,
        'confidence': float(np.max(probabilities)),
        'top_3': top_3,
        'all_probabilities': {emotion: float(prob) for emotion, prob in 
                              zip(model.label_encoder.classes_, probabilities)}
    }

# ===================== 8. EXECUTE TRAINING =====================

if __name__ == "__main__":
    # Train the model
    model, results = main()
    
    # Example usage after training
    print("\n" + "="*60)
    print("EXAMPLE PREDICTIONS")
    print("="*60)
    
    test_texts = [
        "ꯑꯩ ꯌꯥꯝꯅ ꯍꯛꯄ ꯐꯥꯔꯦ",
        "ꯑꯩ ꯅꯪꯕꯨ ꯅꯨꯡꯁꯤ",
        "ꯑꯩ ꯌꯥꯝꯅ ꯁꯥꯎꯕ ꯄꯤꯔꯤ"
    ]
    
    for text in test_texts:
        result = predict_emotion(text)
        print(f"\nText: {result['text']}")
        print(f"Predicted Emotion: {result['emotion']}")
        print(f"Confidence: {result['confidence']:.4f}")
        print("Top 3 predictions:")
        for emotion, prob in result['top_3']:
            print(f"  {emotion}: {prob:.4f}")

ModuleNotFoundError: No module named 'sklearn'